# 04 Photometrie-Stapel

**Eingabe**: kalibrierte FITS-Bilder (je Bild eine WCS) + `field_config.json` aus Notebook 03  
**Ausgabe**: `messtabelle.csv` — eine Zeile je Bild und Stern: Position, Fluss, Fehler, Himmel, Peak, Sättigung, Flags

Beispiel in diesem Notebook: **LCO-Archiv, δ Cep, Filter V, 0,4-m-Teleskop `0m4-11`** (öffentliche Daten, BANZAI-kalibriert).
Eigene Bilder: Pfade und Auswahl in Schritt 1 ändern.

Schritte: 1 Bilder wählen · 2 Sterne laden · 3 Photometrie · 4 Bildqualität (Flags) · 5 speichern

## Schritt 1 — Bilder und Pfade

`FRAMES_DIR` = Ordner mit den Bildern, `TELESCOPE` = optionaler Filter auf `TELESCOP` (hier nur das Teleskop `0m4-11`,
weil die Archivbilder von sechs Teleskopen mit anderer Pixelskala stammen). `FIELD_CONFIG` kommt aus NB03
(`VARPHOT_FRAME=<Referenzbild>` setzt dort Feld und Winkel aus dem Bild).

In [ ]:
import os, json
from pathlib import Path
import pandas as pd
from astropy.io import fits
from tqdm.auto import tqdm

from varphot import (CFG, run_batch, qc_frames, read_measurements, stars_from_field_config)
from varphot.observer import data_dir
from varphot.testdata import testdata_dir

FRAMES_DIR   = Path(os.environ.get("VARPHOT_FRAMES_DIR") or testdata_dir() / "lco-dcep").expanduser()   # Download: tools/download_testdata.py
TELESCOPE    = os.environ.get("VARPHOT_TELESCOPE", "0m4-11")          # "" = alle
FIELD_CONFIG = Path(os.environ.get("VARPHOT_FIELD_CONFIG", "../data/fields/del_Cep_lco_0m4_field_config.json"))
TAG          = os.environ.get("VARPHOT_TAG", "delta_cep_lco_0m4_11")
RESULTS      = (data_dir() if data_dir().is_dir() else Path("../data")) / "results" / TAG
RESULTS.mkdir(parents=True, exist_ok=True)
OUT = RESULTS / "messtabelle.csv"

def _telescope(p):
    with fits.open(p) as h:
        hdr = h["SCI"].header if "SCI" in h else h[0].header
        return str(hdr.get("TELESCOP", ""))

alle = sorted(list(FRAMES_DIR.glob("*.fits.fz")) + list(FRAMES_DIR.glob("*.fits")) + list(FRAMES_DIR.glob("*.fit")))
FRAMES = [p for p in alle if not TELESCOPE or _telescope(p) == TELESCOPE]
print(f"{len(alle)} Bilder im Ordner, {len(FRAMES)} nach Teleskop-Filter '{TELESCOPE}'")
print("Ergebnisordner:", RESULTS)

## Schritt 2 — Sterne laden

Ziel, Vergleichssterne (Comps) und Check-Stern aus `field_config.json`. Gibt NB03 keinen Check-Stern her (kleines
Feld), wird er hier ausdrücklich aus den Comps bestimmt (`ROLLEN`): Er geht dann **nicht** in den Nullpunkt ein.

In [ ]:
record = json.loads(FIELD_CONFIG.read_text())
STARS = stars_from_field_config(record)
ROLLEN = {"g1": "check"}                  # Check-Stern: ein schwächerer Comp, der nicht in den Nullpunkt eingeht
STARS["role"] = [ROLLEN.get(i, r) for i, r in zip(STARS["id"], STARS["role"])]
STARS["id"] = STARS["id"].astype(str)
display(STARS.round(4))

## Schritt 3 — Stapellauf Aperturphotometrie

Je Bild: Positionen aus der WCS des Bildes, Apertur 10 px mit Himmelsring 16–24 px (bei 0,58″/px, Seeing ~2″),
Fluss, Fehler (CCD-Gleichung mit Gain/Ausleserauschen aus dem Header), Himmel, Peak, Zahl gesättigter Pixel.
Läuft parallel; Ergebnis wird zwischengespeichert (`skip_if_fresh`).

In [ ]:
from varphot import skip_if_fresh
CORE_R = float(os.environ.get("VARPHOT_CORE_R", 1.5))   # Kernmaske für gesättigtes Ziel [px]; 0 = aus (siehe Schritt 4b)
AP = {"ap_r": 10, "ap_r_in": 16, "ap_r_out": 24}                 # Standard (LCO-Beispiel); mit Plan aus NB00 überschrieben
if os.environ.get("VARPHOT_PLAN"):
    from varphot import load_plan
    PLAN = load_plan(os.environ["VARPHOT_PLAN"])
    AP = PLAN.aperture or AP
    print(f"Plan {PLAN.target} / {PLAN.method} / {PLAN.camera_model}: Aperturen aus dem Plan {AP}; Meldefilter {PLAN.aavso_filter}")
CFG_RUN = {**CFG, **AP, "core_r": CORE_R or None}
RAW = RESULTS / f"messtabelle_roh_core{CORE_R:g}_ap{AP['ap_r']}-{AP['ap_r_in']:g}-{AP['ap_r_out']}.csv"      # Schlüssel enthält Kernmaske und Aperturen

if skip_if_fresh(RAW, FIELD_CONFIG, *[str(p) for p in FRAMES]):
    tab = read_measurements(RAW)
else:
    tab = run_batch(FRAMES, STARS, CFG_RUN, workers=4, progress=lambda it, total: tqdm(it, total=total))
    tab["star_id"] = tab["star_id"].astype(str)
    tab.to_csv(RAW, index=False)
print(f"{tab['file'].nunique()} Bilder × {tab['star_id'].nunique()} Sterne = {len(tab)} Messungen")

## Schritt 4 — Bildqualität (Flags)

Ein Bild ist **brauchbar**, wenn alle Comps genug Signal haben (`NO_SIGNAL`), die Transparenz stimmt
(`LOW_TRANSPARENCY`, dünne Wolken), das Seeing nicht schlecht ist (`BAD_SEEING`) und eine WCS vorhanden ist (`NO_WCS`).
Die Schwellen sind Startwerte (`snr_min`, `transp_min`, `fwhm_max_arcsec` in `CFG`).

In [ ]:
roles = dict(zip(STARS["id"], STARS["role"]))
tab = qc_frames(tab, roles, CFG_RUN)
fl = tab.drop_duplicates("file")[["file", "frame_flags", "transparency", "fwhm_arcsec"]]
print(f"{(fl['frame_flags'] == '').sum()} von {len(fl)} Bildern brauchbar")
print(fl["frame_flags"].replace("", "brauchbar").value_counts().to_string())

## Schritt 4b — Kernmaske für ein gesättigtes Ziel prüfen

Ist das Ziel gesättigt (hier δ Cep: der Kern steht bei ~103 000 ADU), geht Fluss verloren — und **wie viel**, hängt vom
Seeing ab (schärferes Bild = mehr Sättigung). Das verfälscht die Lichtkurve von Nacht zu Nacht. Gegenmittel: den Fluss nur im
Ring außerhalb des Kerns messen und mit dem Ringanteil heller, ungesättigter Comps desselben Bildes auf den Gesamtfluss
hochrechnen (`light_curve(core_correct=True)`).

Diese Zelle probiert mehrere Kernradien durch und vergleicht das **Residuum um eine Fourier-Reihe** (kleiner = besser) und die
Korrelation mit dem Seeing (soll ≈ 0 sein). `PERIOD_RANGE` = Suchbereich der Periode. Der beste Radius gehört nach `CORE_R`
(Schritt 3) — er wird bewusst von Hand gesetzt, weil er an *diesen* Daten gewählt wird.

In [ ]:
from varphot import light_curve, harmonic_fit
import numpy as np

PERIOD_RANGE = (5.0, 5.8)                 # δ Cep: 5,366 d
good = set(tab.loc[tab["frame_flags"] == "", "file"])
paths = [p for p in FRAMES if p.name in good]
rows = []
for r in [0, 1.0, 1.5, 2.0, 2.5]:
    t_ = run_batch(paths, STARS, {**CFG_RUN, "core_r": r or 1.0}, workers=4)
    t_["star_id"] = t_["star_id"].astype(str)
    t_ = qc_frames(t_, roles, CFG_RUN)
    lc_ = light_curve(t_, STARS, "target", core_correct=bool(r))
    f = harmonic_fit(lc_["jd_mid"], lc_["mag"], *PERIOD_RANGE)
    rows.append(dict(kernradius_px=r or "aus", Residuum_mag=f["rms"], Periode_d=f["period"], Amplitude_mag=f["amplitude"],
                     Korr_FWHM=np.corrcoef(f["residual"], lc_["fwhm_arcsec"])[0, 1]))
display(pd.DataFrame(rows).round(3))

## Schritt 5 — Messtabelle speichern

In [ ]:
tab.to_csv(OUT, index=False)
print("gespeichert:", OUT)
display(tab[tab["star_id"] == "target"][["file", "jd_mid", "flux", "flux_err", "peak", "n_sat", "frame_flags"]].head())